In [0]:
from pyspark.sql import functions as F
 
dim_date = (spark.sql("""
    SELECT explode(sequence(
        to_date("2024-01-01"), to_date("2024-12-31"), interval 1 day)) AS date_key
""")
    .withColumn("year",        F.year("date_key"))
    .withColumn("month",       F.month("date_key"))
    .withColumn("day",         F.dayofmonth("date_key"))
    .withColumn("day_of_week", F.dayofweek("date_key"))
    .withColumn("day_name",    F.date_format("date_key", "EEEE"))
    .withColumn("month_name",  F.date_format("date_key", "MMMM"))
    .withColumn("quarter",     F.quarter("date_key"))
    .withColumn("is_weekend",  F.dayofweek("date_key").isin([1, 7]))
)
 
dim_date.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.dim_date")

In [0]:
zones = spark.table("nyc_taxi.silver.dim_taxi_zone_raw")
 
dim_zone = zones.select(
    F.col("LocationID").cast("int").alias("zone_key"),
    F.col("Borough").alias("borough"),
    F.col("Zone").alias("zone_name"),
    F.col("service_zone"))
 
# The -1 Unknown row: every dimension needs one
unknown = spark.createDataFrame(
    [(-1, "Unknown", "Unknown", "Unknown")],
    "zone_key int, borough string, zone_name string, service_zone string")
 
dim_zone = dim_zone.unionByName(unknown)
dim_zone.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.dim_taxi_zone")


In [0]:
rate_codes = spark.createDataFrame([
    (1, "Standard rate"), (2, "JFK"), (3, "Newark"),
    (4, "Nassau or Westchester"), (5, "Negotiated fare"),
    (6, "Group ride"), (99, "Unknown"), (-1, "Unknown")],
    "rate_code_key int, rate_code_desc string")
rate_codes.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.dim_rate_code")
 
payment_types = spark.createDataFrame([
    (1, "Credit card"), (2, "Cash"), (3, "No charge"), (4, "Dispute"),
    (5, "Unknown"), (6, "Voided trip"), (-1, "Unknown")],
    "payment_type_key int, payment_type_desc string")
payment_types.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.gold.dim_payment_type")


In [0]:
silver = spark.table("nyc_taxi.silver.trips_clean")
zone_keys = [r.zone_key for r in spark.table("nyc_taxi.gold.dim_taxi_zone").select("zone_key").collect()]
 
fact = (silver
    .withColumn("date_key",   F.col("pickup_date"))
    .withColumn("pickup_zone_key",
        F.when(F.col("PULocationID").isin(zone_keys), F.col("PULocationID")).otherwise(F.lit(-1)))
    .withColumn("dropoff_zone_key",
        F.when(F.col("DOLocationID").isin(zone_keys), F.col("DOLocationID")).otherwise(F.lit(-1)))
    .withColumn("rate_code_key",
        F.coalesce(F.col("RatecodeID").cast("int"), F.lit(-1)))
    .withColumn("payment_type_key",
        F.coalesce(F.col("payment_type").cast("int"), F.lit(-1)))
    .select(
        "trip_id", "date_key", "pickup_zone_key", "dropoff_zone_key",
        "rate_code_key", "payment_type_key",
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance", "trip_duration_min",
        "fare_amount", "tip_amount", "total_amount")
)
 
(fact.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("date_key")
    .saveAsTable("nyc_taxi.gold.fact_trips"))

In [0]:
%sql
CREATE OR REPLACE VIEW nyc_taxi.gold.dim_pickup_zone AS
SELECT
  zone_key AS pickup_zone_key,
  borough AS pickup_borough,
  zone_name AS pickup_zone_name,
  service_zone AS pickup_service_zone
FROM nyc_taxi.gold.dim_taxi_zone;

CREATE OR REPLACE VIEW nyc_taxi.gold.dim_dropoff_zone AS
SELECT
  zone_key AS dropoff_zone_key,
  borough AS dropoff_borough,
  zone_name AS dropoff_zone_name,
  service_zone AS dropoff_service_zone
FROM nyc_taxi.gold.dim_taxi_zone;

In [0]:
%sql
SELECT table_schema, table_name, table_type
FROM nyc_taxi.information_schema.tables
ORDER BY table_schema, table_type, table_name;